# Lesson 6 — Production Polish: Validated Outputs + Question Bank + Trace Toggle

**Audience:** University faculty (Malaysia)  
**Outcome:** This lesson turns our multi-agent system into something that feels *usable in real university workflows*.

---

## What we add in Lesson 6

### ✅ A shared output contract (validated)
We standardize all results into:
- **Human-readable Markdown**
- **Machine-readable JSON** (validated with **Pydantic**)

### ✅ Trace mode toggle
- `TRACE_MODE=1` → show all agent inputs/outputs  
- `TRACE_MODE=0` → show only the final answer (clean UI)

### ✅ Auto-generated question bank
We generate realistic faculty questions from:
- the **Course Policy Handbook** (Doc QA agent)
- the **Campus dataset** (MCP tools agent)

---

## Prerequisites (start these servers in terminals)

Terminal 1:
```bash
python a2a_course_policy_agent.py
```

Terminal 2:
```bash
python a2a_campus_info_agent.py
```

Terminal 3:
```bash
python a2a_web_research_agent.py
```

> Notebook tip: we use top-level `await` (not `asyncio.run`).


## 1. Setup

Install (if needed):
- `pydantic` (validation)
- `openai` (LLM router + question generation)
- `httpx` + `a2a-sdk` client (calling agents)
- `pypdf` (extract a few policy snippets for question generation)


In [1]:
# If needed, uncomment and run:
!pip -q install pydantic openai httpx a2a-sdk python-dotenv pypdf

In [6]:
import openai
print(openai.__version__)

1.83.0


In [2]:
import os
from dotenv import load_dotenv

_ = load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your environment."
print("✅ OPENAI_API_KEY found")

✅ OPENAI_API_KEY found


## 2. Load the two content sources (for question generation)

We will generate questions from:
1) `SENG3200_Course_Policy_Handbook.pdf`
2) `campus_data.json`

These are the same sources used by:
- CoursePolicyAgent (PDF grounded)
- CampusInfoToolAgent (MCP tools + JSON data)


In [ ]:
import json
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("/mnt/data/SENG3200_Course_Policy_Handbook.pdf")
CAMPUS_JSON = Path("/mnt/data/campus_data.json")

assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"
assert CAMPUS_JSON.exists(), f"Missing JSON: {CAMPUS_JSON}"

campus_data = json.loads(CAMPUS_JSON.read_text())

reader = PdfReader(str(PDF_PATH))
policy_pages = []
for i, page in enumerate(reader.pages[:6], start=1):
    text = (page.extract_text() or "").strip()
    policy_pages.append(f"[Page {i}] " + " ".join(text.split()))

policy_excerpt = "\n\n".join(policy_pages)

print("✅ Loaded policy excerpt chars:", len(policy_excerpt))
print("✅ Loaded campus staff:", len(campus_data["staff"]))

## 3. Define a validated output contract (Pydantic)

Every final result must validate to this schema:

- `answer`: short, directly useful
- `evidence`: structured list of where the answer came from (policy/tool/web)
- `actions`: recommended next steps
- `confidence`: high/medium/low
- `trace` (optional): per-agent inputs/outputs (shown only in TRACE_MODE)


In [3]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

EvidenceSource = Literal["policy_doc", "campus_tool", "web_research"]

class EvidenceItem(BaseModel):
    source: EvidenceSource
    ref: str = Field(..., description="Short reference e.g., page number, tool name, or URL")
    detail: str = Field(..., description="Short supporting detail")

class AgentTrace(BaseModel):
    agent_name: str
    agent_url: str
    prompt: str
    response_text: str

class FinalResponse(BaseModel):
    answer: str
    evidence: List[EvidenceItem]
    actions: List[str]
    confidence: Literal["high", "medium", "low"]
    trace: Optional[List[AgentTrace]] = None

print("✅ Pydantic schema ready")

✅ Pydantic schema ready


## 4. A2A client helper (call any agent + capture trace)

In [4]:
import httpx
from typing import Dict, Any

from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.types import AgentCard, Artifact, Message, Task
from a2a.utils.message import get_message_text

async def call_a2a_agent(prompt: str, base_url: str) -> Dict[str, Any]:
    async with httpx.AsyncClient(timeout=180.0) as httpx_client:
        client = await ClientFactory.connect(
            base_url,
            client_config=ClientConfig(httpx_client=httpx_client),
        )
        card: AgentCard = await client.get_card()

        message = create_text_message_object(content=prompt)
        responses = client.send_message(message)

        text_content = ""
        async for response in responses:
            if isinstance(response, Message):
                text_content = get_message_text(response)
            elif isinstance(response, tuple):
                task: Task = response[0]
                if task.artifacts:
                    artifact: Artifact = task.artifacts[0]
                    text_content = get_message_text(artifact)

        return {
            "agent_name": card.name,
            "agent_url": card.url,
            "prompt": prompt,
            "response_text": text_content,
        }

## 5. LLM Router (JSON Schema output)

We use an LLM router to:
- decide which agent(s) to call
- produce *agent-specific prompts* (decomposition)

✅ We use JSON Schema output so it is always parseable.


In [9]:
from openai import OpenAI
import os, json
from IPython.display import Markdown, display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

POLICY_URL = "http://127.0.0.1:9999"
CAMPUS_URL = "http://127.0.0.1:9998"
RESEARCH_URL = "http://127.0.0.1:9997"

ROUTER_MODEL = os.environ.get("ROUTER_MODEL", "gpt-4o-mini")

ROUTER_SCHEMA = {
  "type": "object",
  "properties": {
    "policy": {"type": "boolean"},
    "campus": {"type": "boolean"},
    "research": {"type": "boolean"},
    "policy_prompt": {"type": ["string", "null"]},
    "campus_prompt": {"type": ["string", "null"]},
    "research_prompt": {"type": ["string", "null"]},
    "why": {"type": "string"}
  },
  "required": ["policy","campus","research","policy_prompt","campus_prompt","research_prompt","why"],
  "additionalProperties": False
}

def build_plan(question: str) -> dict:
    resp = client.responses.create(
        model=ROUTER_MODEL,
        input=[
            {"role": "system", "content": ROUTER_SYSTEM},
            {"role": "user", "content": f"USER QUESTION:\n{question}"},
        ],
        # ✅ Correct for Responses API structured outputs in this SDK:
        text={
            "format": {
                "type": "json_schema",
                "name": "router_output",
                "strict": True,
                "schema": ROUTER_SCHEMA,
            }
        },
    )

    route = json.loads(resp.output_text)

    calls = []
    if route["policy"]:
        calls.append({"agent": "CoursePolicyAgent", "url": POLICY_URL, "prompt": route["policy_prompt"]})
    if route["campus"]:
        calls.append({"agent": "CampusInfoToolAgent", "url": CAMPUS_URL, "prompt": route["campus_prompt"]})
    if route["research"]:
        calls.append({"agent": "WebResearchAgent", "url": RESEARCH_URL, "prompt": route["research_prompt"]})

    return {"question": question, "route": route, "calls": calls}

demo_plan = build_plan("Where is CS-Lab-2 and what facilities does it have?")

display(Markdown(f"""### Router demo
- Policy: **{demo_plan['route']['policy']}**
- Campus: **{demo_plan['route']['campus']}**
- Research: **{demo_plan['route']['research']}**

**Why:** {demo_plan['route']['why']}

**Calls:** {', '.join([c['agent'] for c in demo_plan['calls']])}
"""))

### Router demo
- Policy: **False**
- Campus: **True**
- Research: **False**

**Why:** The question is about the location and facilities of a specific lab, which falls under campus information.

**Calls:** CampusInfoToolAgent


## 6. Response normalization: parse agent outputs into our schema

In [10]:
import re
import json as pyjson
from typing import Tuple

def extract_section(text: str, header: str) -> str:
    if not text:
        return ""
    pattern = rf"{header}\s*:\s*(.*?)(?=\n[A-Z][A-Za-z /]*\s*:|\Z)"
    m = re.search(pattern, text, flags=re.S | re.M)
    return m.group(1).strip() if m else ""

def compact(text: str, max_len: int = 500) -> str:
    t = " ".join((text or "").split())
    return (t[:max_len] + "…") if len(t) > max_len else t

def normalize_from_policy(text: str) -> Tuple[str, list, list, str]:
    ans = extract_section(text, "Answer") or text
    ev = extract_section(text, "Evidence")
    actions = extract_section(text, "Actions / Next steps")

    evidence_items = []
    for line in (ev.splitlines() if ev else []):
        line = line.strip("- ").strip()
        if not line:
            continue
        m = re.search(r"(Page\s*\d+)", line)
        ref = m.group(1) if m else "policy"
        evidence_items.append(EvidenceItem(source="policy_doc", ref=ref, detail=compact(line, 180)))

    action_list = [a.strip("- ").strip() for a in actions.splitlines() if a.strip()] if actions else []
    if not action_list:
        action_list = ["Verify the handbook section/page reference in the LMS copy."]

    confidence = "high" if evidence_items else "medium"
    return compact(ans, 700), evidence_items, action_list, confidence

def normalize_from_campus(text: str) -> Tuple[str, list, list, str]:
    ans = extract_section(text, "Answer") or text
    ev = extract_section(text, "Evidence")
    actions = extract_section(text, "Actions / Next steps")

    tool_name = None
    if ev:
        for line in ev.splitlines():
            if "Tool:" in line:
                tool_name = line.split("Tool:", 1)[1].strip()

    evidence_items = [EvidenceItem(source="campus_tool", ref=tool_name or "campus_tool", detail="Result from MCP campus tool output.")]

    action_list = [a.strip("- ").strip() for a in actions.splitlines() if a.strip()] if actions else []
    if not action_list:
        action_list = ["If the record is missing, confirm the name/room code and try again."]

    return (ans.strip() if ans else ""), evidence_items, action_list, "high"

def normalize_from_research(text: str) -> Tuple[str, list, list, str]:
    ans = extract_section(text, "Answer") or text
    sources_block = extract_section(text, "Sources")
    evidence_items = []

    if sources_block:
        for line in sources_block.splitlines():
            line = line.strip("- ").strip()
            if not line:
                continue
            if "—" in line:
                title, url = [p.strip() for p in line.split("—", 1)]
                evidence_items.append(EvidenceItem(source="web_research", ref=url, detail=compact(title, 160)))
            else:
                evidence_items.append(EvidenceItem(source="web_research", ref="source", detail=compact(line, 160)))

    actions = ["Prefer official university/academic sources; verify policy applicability before publishing."]
    confidence = "medium" if evidence_items else "low"
    return compact(ans, 900), evidence_items, actions, confidence

print("✅ Normalizers ready")

✅ Normalizers ready


## 7. Orchestrator (polished): returns validated JSON + clean Markdown

We respect `TRACE_MODE`:
- if `TRACE_MODE=1`, include agent traces
- else return only clean final output


In [11]:
TRACE_MODE = os.environ.get("TRACE_MODE", "1") == "1"

def is_policy_not_found(text: str) -> bool:
    return (text or "").strip() == "Not found in the provided document."

def campus_failed(text: str) -> bool:
    t = (text or "").lower()
    return ("error" in t) or ("room not found" in t) or ("no office hours found" in t)

async def run_orchestrator(question: str) -> FinalResponse:
    plan = build_plan(question)
    traces_raw = []

    for c in plan["calls"]:
        traces_raw.append(await call_a2a_agent(c["prompt"], c["url"]))

    already_research = any(t["agent_name"] == "WebResearchAgent" for t in traces_raw)

    for t in traces_raw:
        if t["agent_name"] == "CoursePolicyAgent" and is_policy_not_found(t["response_text"]) and not already_research:
            traces_raw.append(await call_a2a_agent(question, RESEARCH_URL))
            already_research = True
            break
        if t["agent_name"] == "CampusInfoToolAgent" and campus_failed(t["response_text"]) and not already_research:
            traces_raw.append(await call_a2a_agent(question, RESEARCH_URL))
            already_research = True
            break

    answer_parts = []
    evidence = []
    actions = []
    confidences = []

    for t in traces_raw:
        name = t["agent_name"]
        if name == "CoursePolicyAgent":
            a, e, act, conf = normalize_from_policy(t["response_text"])
            if a:
                answer_parts.append(f"**Course policy:** {a}")
            evidence.extend(e)
            actions.extend(act)
            confidences.append(conf)

        elif name == "CampusInfoToolAgent":
            a, e, act, conf = normalize_from_campus(t["response_text"])
            if a:
                answer_parts.append(f"**Campus info:**\n{a}")
            evidence.extend(e)
            actions.extend(act)
            confidences.append(conf)

        elif name == "WebResearchAgent":
            a, e, act, conf = normalize_from_research(t["response_text"])
            if a:
                answer_parts.append(f"**Web research:** {a}")
            evidence.extend(e)
            actions.extend(act)
            confidences.append(conf)

    actions = list(dict.fromkeys([a for a in actions if a]))

    confidence = "medium"
    if "high" in confidences and len(evidence) >= 2:
        confidence = "high"
    if all(c == "low" for c in confidences):
        confidence = "low"

    trace_models = None
    if TRACE_MODE:
        trace_models = [AgentTrace(**t) for t in traces_raw]

    final = FinalResponse(
        answer="\n\n".join(answer_parts).strip() or "No answer returned.",
        evidence=evidence[:10],
        actions=actions[:8],
        confidence=confidence,
        trace=trace_models,
    )
    return final

from IPython.display import Markdown, display

def render_markdown(final: FinalResponse, question: str) -> None:
    md = []
    md.append("# ✅ Final Output (Polished)")
    md.append(f"**Question:** `{question}`\n")
    md.append("## Answer")
    md.append(final.answer)
    md.append("\n## Evidence")
    for ev in final.evidence:
        md.append(f"- **{ev.source}** ({ev.ref}): {ev.detail}")
    md.append("\n## Actions / Next steps")
    for a in final.actions:
        md.append(f"- {a}")
    md.append(f"\n**Confidence:** `{final.confidence}`")

    if final.trace:
        md.append("\n---\n## Trace (Agent Inputs/Outputs)")
        for t in final.trace:
            md.append(f"### {t.agent_name}")
            md.append("**Input**")
            md.append(f"```text\n{t.prompt}\n```")
            md.append("**Output**")
            md.append(f"```text\n{t.response_text}\n```")

    display(Markdown("\n".join(md)))

def render_json(final: FinalResponse) -> None:
    display(Markdown("## JSON (Validated)"))
    print(final.model_dump_json(indent=2))

print("✅ Orchestrator polish ready. TRACE_MODE =", TRACE_MODE)

✅ Orchestrator polish ready. TRACE_MODE = True


## 8. Demo: run polished orchestrator (2 examples)

> If a server is not running, this will fail. Start the terminals first.


In [12]:
question = "What is the late penalty after 2 days, and what are Dr Amina Rahman's office hours?"
final = await run_orchestrator(question)
render_markdown(final, question)
render_json(final)

# ✅ Final Output (Polished)
**Question:** `What is the late penalty after 2 days, and what are Dr Amina Rahman's office hours?`

## Answer
**Course policy:** For submissions that are 2 days late, a penalty of 20% is deducted, resulting in a maximum of 80% of the total marks available.

**Campus info:**
- I cannot find Dr. Amina Rahman's office hours.

## Evidence
- **policy_doc** (Page 4): "2 days (25–48 hrs) 20% deducted 80% of total marks" (Page 4)
- **policy_doc** (Page 4): "Penalties are non-negotiable after the fact." (Page 4)
- **policy_doc** (Page 4): "A 'day' is defined as any 24-hour period or part thereof after the deadline." (Page 4)
- **campus_tool** (get_office_hours): Result from MCP campus tool output.

## Actions / Next steps
- Verify the handbook section/page reference in the LMS copy.
- You may want to check the university's faculty directory or contact the department directly for more information.

**Confidence:** `high`

---
## Trace (Agent Inputs/Outputs)
### CoursePolicyAgent
**Input**
```text
What is the late penalty for submissions after 2 days?
```
**Output**
```text
Answer: For submissions that are 2 days late, a penalty of 20% is deducted, resulting in a maximum of 80% of the total marks available.

Evidence:
- "2 days (25–48 hrs) 20% deducted 80% of total marks" (Page 4)
- "Penalties are non-negotiable after the fact." (Page 4)
- "A 'day' is defined as any 24-hour period or part thereof after the deadline." (Page 4)
```
### CampusInfoToolAgent
**Input**
```text
What are Dr. Amina Rahman's office hours?
```
**Output**
```text
Answer:
- I cannot find Dr. Amina Rahman's office hours.

Evidence:
- Tool: get_office_hours
- Key fields: staff_name: Dr. Amina Rahman

Actions / Next steps:
- You may want to check the university's faculty directory or contact the department directly for more information.
```

## JSON (Validated)

{
  "answer": "**Course policy:** For submissions that are 2 days late, a penalty of 20% is deducted, resulting in a maximum of 80% of the total marks available.\n\n**Campus info:**\n- I cannot find Dr. Amina Rahman's office hours.",
  "evidence": [
    {
      "source": "policy_doc",
      "ref": "Page 4",
      "detail": "\"2 days (25–48 hrs) 20% deducted 80% of total marks\" (Page 4)"
    },
    {
      "source": "policy_doc",
      "ref": "Page 4",
      "detail": "\"Penalties are non-negotiable after the fact.\" (Page 4)"
    },
    {
      "source": "policy_doc",
      "ref": "Page 4",
      "detail": "\"A 'day' is defined as any 24-hour period or part thereof after the deadline.\" (Page 4)"
    },
    {
      "source": "campus_tool",
      "ref": "get_office_hours",
      "detail": "Result from MCP campus tool output."
    }
  ],
  "actions": [
    "Verify the handbook section/page reference in the LMS copy.",
    "You may want to check the university's faculty directory or cont

In [ ]:
question = "Where is CS-Lab-2 and what facilities does it have?"
final = await run_orchestrator(question)
render_markdown(final, question)
render_json(final)

## 9. Auto-generate a question bank (from the two content sources)

We generate 12 realistic questions:
- 6 policy questions
- 6 campus questions

Each question includes:
- expected_agent: policy | campus
- short_expected_answer (1 sentence)

This is useful for:
- demos
- professor exercises
- later evaluation/testing


In [ ]:
from typing import TypedDict, Literal, List
from openai import OpenAI

qg_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
QG_MODEL = os.environ.get("QG_MODEL", "gpt-4o-mini")

class QuestionItem(TypedDict):
    question: str
    expected_agent: Literal["policy", "campus"]
    short_expected_answer: str

def generate_question_bank(policy_text: str, campus_json: dict) -> List[QuestionItem]:
    prompt = f"""You are generating training questions for university faculty.

Create:
- 6 questions answerable from the COURSE POLICY excerpt.
- 6 questions answerable from the CAMPUS JSON dataset.

Return ONLY valid JSON: a list of 12 objects.
Each object MUST have:
- question (string)
- expected_agent ("policy" or "campus")
- short_expected_answer (string, 1 sentence)

COURSE POLICY EXCERPT:
{policy_text[:3500]}

CAMPUS JSON:
{pyjson.dumps(campus_json, indent=2)[:3500]}
"""

    resp = qg_client.responses.create(
        model=QG_MODEL,
        input=prompt,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "question_bank",
                "schema": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "question": {"type": "string"},
                            "expected_agent": {"type": "string", "enum": ["policy", "campus"]},
                            "short_expected_answer": {"type": "string"},
                        },
                        "required": ["question", "expected_agent", "short_expected_answer"],
                    },
                    "minItems": 12,
                    "maxItems": 12,
                },
            },
        },
    )
    return resp.output_parsed

question_bank = generate_question_bank(policy_excerpt, campus_data)
len(question_bank), question_bank[0]

In [ ]:
from IPython.display import Markdown, display

md = ["# 🧪 Generated Question Bank (12)"]
for i, item in enumerate(question_bank, start=1):
    md.append(f"### Q{i}. ({item['expected_agent']}) {item['question']}")
    md.append(f"- **Expected (short):** {item['short_expected_answer']}")
display(Markdown("\n".join(md)))

## 10. Quick run: execute 4 generated questions through the orchestrator

We pick:
- 2 policy questions
- 2 campus questions

This shows:
- the router decision
- validated JSON output
- consistent formatting


In [ ]:
sample = []
for item in question_bank:
    if item["expected_agent"] == "policy" and len([x for x in sample if x["expected_agent"]=="policy"]) < 2:
        sample.append(item)
    if item["expected_agent"] == "campus" and len([x for x in sample if x["expected_agent"]=="campus"]) < 2:
        sample.append(item)
    if len(sample) == 4:
        break

for item in sample:
    q = item["question"]
    print("\n" + "="*110)
    print("Question:", q, "| Expected agent:", item["expected_agent"])
    final = await run_orchestrator(q)
    render_markdown(final, q)

## 11. Exercises (for faculty participants)

1) Turn TRACE_MODE off:
```bash
export TRACE_MODE=0
```
Restart kernel and re-run demos.

2) Add an automated check:
- compare `expected_agent` with the router's choice
- log mismatches

3) Improve evidence:
- policy: add explicit page numbers in evidence items
- campus: add tool name + key fields
